In [ ]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
xbows = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b][: t+1]
        xprev_mean = torch.mean(xprev, dim=0)
        xbows[b][t] = xprev_mean
print(x[0])
print(xbows[0])


tensor([[-0.3171,  2.2151],
        [ 0.7151, -0.9198],
        [-1.0873,  0.6710],
        [-0.8072, -0.1908],
        [ 0.4269, -0.4222],
        [-0.7732,  0.7583],
        [-0.6962,  0.0648],
        [ 1.0424, -1.6809]])
tensor([[-0.3171,  2.2151],
        [ 0.1990,  0.6477],
        [-0.2298,  0.6554],
        [-0.3741,  0.4439],
        [-0.2139,  0.2707],
        [-0.3071,  0.3519],
        [-0.3627,  0.3109],
        [-0.1871,  0.0619]])


In [4]:
tril = torch.tril(torch.ones(T, T))
tril_sum = tril.sum(1, keepdim=True)
wei = tril / tril_sum 
print(wei)
xbow2 = wei @ x # wei: (T, T), x: (B, T, C)
print(torch.allclose(xbows, xbow2))

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])
True


In [5]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
print(wei)
dropout = nn.Dropout(0.2)
wei = dropout(wei)
print(wei)
xbow3 = wei @ x
print(torch.allclose(xbow3, xbow2))

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])
tensor([[1.2500, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.6250, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4167, 0.4167, 0.4167, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3125, 0.3125, 0.3125, 0.3125, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.0000, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000],
        [0.2083, 0.2083, 0.2083, 0.20